# 04 LEAR FS1

This notebook focuses on the LEAR model with `FS1` features only: lagged day-ahead prices.

Because the saved LEAR benchmark reruns `FS1` and `FS2` together for fairness, the optional rerun cell below reproduces the **full LEAR benchmark suite** and then this notebook filters to the `FS1` view.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig, MonitoringConfig
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.notebook_support import (
    build_week_metrics_for_predictions,
    build_reporting_metric_grid,
    estimate_run_duration_seconds,
    format_duration,
    load_selected_case_weeks,
    render_plot_gallery,
    render_reporting_metric_dashboard,
    run_suite_with_feedback,
    style_reporting_metric_grid,
    summarize_timing_compact,
    write_week_plots_for_models,
)

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
run_root = output_root / "runs"


def latest_run_matching(pattern: str) -> Path:
    matches = sorted(run_root.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No run directories found for pattern: {pattern}")
    return matches[-1]


The saved LEAR benchmark reruns both feature sets together for fairness. This notebook filters to the `FS1` view after loading the shared benchmark artifact.


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    from dataclasses import replace

    from dataclasses import replace
    from hourly_da.models.lear import LEARModel, LEARSettings
    from hourly_da.models.naive import PreviousWeekNaiveModel, PreviousYearNaiveModel
    rerun_config = replace(config)

    rerun_config = replace(config, monitoring=MonitoringConfig(fit_time_absolute_threshold_sec=5.0))
    models = [
        PreviousWeekNaiveModel(),
        PreviousYearNaiveModel(),
        LEARModel(LEARSettings(fs_level="FS1", training_window_hours=90 * 24, min_train_rows=30 * 24, alpha=0.01)),
        LEARModel(LEARSettings(fs_level="FS2", training_window_hours=90 * 24, min_train_rows=30 * 24, alpha=0.01)),
    ]
    rerun_payload = run_suite_with_feedback(
        config=rerun_config,
        run_label="lear_benchmark",
        models=models,
        include_external_features=False,
        show_progress=True,
        progress_label="lear_benchmark",
    )
    print(rerun_payload["official_naive"])
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True to execute the shared benchmark pipeline from this notebook.")


In [ ]:
run_dir = find_latest_run(output_root, "lear_benchmark")
print(run_dir)

metrics_overall = load_csv(run_dir, "metrics_overall.csv")
metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
predictions_long = load_csv(run_dir, "predictions_long.csv")
timing_summary = load_csv(run_dir, "origin_timing_summary.csv")
official_naive = load_json(run_dir, "official_naive_reference.json")
diebold_mariano_by_reporting_level = load_csv(run_dir, "diebold_mariano_by_reporting_level.csv")


In [ ]:
focus_models = ['naive_previous_week', 'naive_previous_year', 'lear_fs1']

display(
    metrics_by_reporting_level[metrics_by_reporting_level["model"].isin(focus_models)]
    .sort_values(["dataset_split", "reporting_level_sort_order", "mae", "model"])
    .reset_index(drop=True)
)

display(
    metrics_overall[metrics_overall["model"].isin(focus_models)]
    .sort_values(["dataset_split", "mae", "model"])
    .reset_index(drop=True)
)

print("Official naive reference:")
display(pd.DataFrame([official_naive]))


dm_reporting = load_csv(run_dir, "diebold_mariano_by_reporting_level.csv")
display(
    dm_reporting[dm_reporting["challenger_model"].isin(['lear_fs1'])]
    .sort_values(["dataset_split", "reporting_level_sort_order", "challenger_model"])
    .reset_index(drop=True)
)


display(timing_summary.sort_values(["dataset_split", "model"]).reset_index(drop=True))


In [ ]:
selection_run_dir, selected_weeks = load_selected_case_weeks(output_root)
print(selection_run_dir)
display(selected_weeks[["category", "iso_week_id", "week_start_local_date", "week_end_local_date"]])

week_metrics = build_week_metrics_for_predictions(predictions_long, config, selected_weeks)
display(
    week_metrics[week_metrics["model"].isin(['naive_previous_week', 'lear_fs1'])]
    .sort_values(["category", "mae", "model"])
    .reset_index(drop=True)
)

plot_dir = output_root / "notebook_artifacts" / "04_lear_fs1" / run_dir.name if "run_dir" in globals() else output_root / "notebook_artifacts" / "04_lear_fs1" / phase_run_dir.name
plot_paths = write_week_plots_for_models(
    predictions=predictions_long,
    config=config,
    selected_weeks=selected_weeks,
    output_dir=plot_dir,
    models=['naive_previous_week', 'lear_fs1'],
    title_prefix="LEAR FS1",
)
display(render_plot_gallery(plot_paths, columns=2))
